# Import Libraries

In [6]:
import pandas as pd
import matplotlib.pyplot as plt

In [24]:
womens_world_cup_stats = pd.read_csv(
    '/Users/gui/womens_world_cup_2023_statsbomb_finalfinal.csv'
)

# Official Rankings & Performance 

In [9]:
official_rankings = {
    "A": ["Switzerland", "Norway", "New Zealand", "Philippines"],
    "B": ["Australia", "Nigeria", "Canada", "Ireland"],
    "C": ["Japan", "Spain", "Zambia", "Costa Rica"],
    "D": ["England", "Denmark", "China", "Haiti"],
    "E": ["Netherlands", "USA", "Portugal", "Vietnam"],
    "F": ["France", "Jamaica", "Brazil", "Panama"],
    "G": ["Sweden", "South Africa", "Italy", "Argentina"],
    "H": ["Colombia", "Morocco", "Germany", "South Korea"]
}

performance_rankings = {
    "A": ["Norway", "Switzerland", "New Zealand", "Philippines"],
    "B": ["Australia", "Nigeria", "Canada", "Ireland"],
    "C": ["Japan", "Spain", "Zambia", "Costa Rica"],
    "D": ["England", "Denmark", "Haiti", "China"],
    "E": ["Netherlands", "USA", "Portugal", "Vietnam"],
    "F": ["France", "Brazil", "Jamaica", "Panama"],
    "G": ["Sweden", "South Africa", "Italy", "Argentina"],
    "H": ["Colombia", "Germany", "Morocco", "South Korea"]
}

# Final Dataset

In [11]:
rows = []

for group in official_rankings:
    for rank in range(4):
        team = official_rankings[group][rank]
        performance_rank = performance_rankings[group].index(team) + 1
        
        rows.append({
            "Group": group,
            "Team": team,
            "Official_Rank": rank + 1,
            "Performance_Rank": performance_rank
        })

group_rankings = pd.DataFrame(rows)

group_rankings

,Group,Team,Official_Rank,Performance_Rank
0,A,Switzerland,1,2
1,A,Norway,2,1
2,A,New Zealand,3,3
3,A,Philippines,4,4
4,B,Australia,1,1
5,B,Nigeria,2,2
6,B,Canada,3,3
7,B,Ireland,4,4
8,C,Japan,1,1
9,C,Spain,2,2


## Diference between rankings 

In [13]:
group_rankings["Rank_Difference"] = (
    group_rankings["Official_Rank"] -
    group_rankings["Performance_Rank"]
)

group_rankings

,Group,Team,Official_Rank,Performance_Rank,Rank_Difference
0,A,Switzerland,1,2,-1
1,A,Norway,2,1,1
2,A,New Zealand,3,3,0
3,A,Philippines,4,4,0
4,B,Australia,1,1,0
5,B,Nigeria,2,2,0
6,B,Canada,3,3,0
7,B,Ireland,4,4,0
8,C,Japan,1,1,0
9,C,Spain,2,2,0


In [14]:
group_rankings["Movement"] = group_rankings["Rank_Difference"].apply(
    lambda x: "Up" if x > 0 else "Down" if x < 0 else "Same"
)

group_rankings

,Group,Team,Official_Rank,Performance_Rank,Rank_Difference,Movement
0,A,Switzerland,1,2,-1,Down
1,A,Norway,2,1,1,Up
2,A,New Zealand,3,3,0,Same
3,A,Philippines,4,4,0,Same
4,B,Australia,1,1,0,Same
5,B,Nigeria,2,2,0,Same
6,B,Canada,3,3,0,Same
7,B,Ireland,4,4,0,Same
8,C,Japan,1,1,0,Same
9,C,Spain,2,2,0,Same


In [15]:
group_stage = womens_world_cup_stats[
    womens_world_cup_stats["stage"] == "Group"
].copy()

group_stage.shape

(96, 53)

In [26]:
group_stage = pd.concat(
    [group_stage],
    ignore_index=True
).copy()

In [28]:
def min_max_score(series):
    if series.max() == series.min():
        return 50
    return ((series - series.min()) / (series.max() - series.min())) * 100

# CREATE THE SCORE OF THE 5 METRICS

In [33]:
group_stage["xG_score"] = min_max_score(
    group_stage["xG"]
)

group_stage["pass_accuracy_score"] = min_max_score(
    group_stage["pass_accuracy_pct"]
)

group_stage["possession_score"] = min_max_score(
    group_stage["possession_pct"]
)

group_stage["goals_scored_score"] = min_max_score(
    group_stage["goals_for"]
)

group_stage["goals_conceded_score"] = (
    100 - min_max_score(group_stage["goals_against"])
)

# PERFORMANCE SCORE

In [36]:
group_stage["performance_score"] = (
    group_stage["xG_score"] * 0.25 +
    group_stage["pass_accuracy_score"] * 0.25 +
    group_stage["possession_score"] * 0.10 +
    group_stage["goals_scored_score"] * 0.25 +
    group_stage["goals_conceded_score"] * 0.15
)

In [38]:
group_stage["performance_score"] = group_stage["performance_score"].round(1)

# FINAL TABLE

In [41]:
performance_table = group_stage[
    [
        "team",
        "xG",
        "pass_accuracy_pct",
        "possession_pct",
        "goals_for",
        "goals_against",
        "performance_score"
    ]
].copy()

performance_table = performance_table.sort_values(
    "performance_score",
    ascending=False
).reset_index(drop=True)



performance_table.insert(
    0,
    "Rank",
    range(1, len(performance_table) + 1)
)


performance_table["xG"] = (
    performance_table["xG"].round(1)
)

performance_table["pass_accuracy_pct"] = (
    performance_table["pass_accuracy_pct"].round(1)
)

performance_table["possession_pct"] = (
    performance_table["possession_pct"].round(1)
)


performance_table = performance_table.rename(columns={
    "team": "Team",
    "xG": "xG (%)",
    "pass_accuracy_pct": "Pass Accuracy (%)",
    "possession_pct": "Possession (%)",
    "goals_for": "Goals Scored",
    "goals_against": "Goals Conceded",
    "performance_score": "Performance Score"
})



performance_table

,Rank,Team,xG (%),Pass Accuracy (%),Possession (%),Goals Scored,Goals Conceded,Performance Score
0,1,Netherlands Women's,4.2,87.1,73.2,7,0,95.7
1,2,Norway Women's,3.5,83.2,69.6,6,0,85.0
2,3,Japan Women's,4.6,79.4,67.9,5,0,85.0
3,4,Spain Women's,3.8,85.6,68.8,5,0,84.4
4,5,Spain Women's,4.2,84.1,83.8,3,0,80.7
...,...,...,...,...,...,...,...,...
91,92,Morocco Women's,0.2,62.8,29.7,0,6,14.3
92,93,Vietnam Women's,0.0,52.7,28.6,0,3,13.3
93,94,Vietnam Women's,0.3,62.9,26.8,0,7,12.3
94,95,Philippines Women's,0.1,56.8,30.4,0,6,10.4


In [48]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

performance_table

,Rank,Team,xG (%),Pass Accuracy (%),Possession (%),Goals Scored,Goals Conceded,Performance Score
0,1,Netherlands Women's,4.2,87.1,73.2,7,0,95.7
1,2,Norway Women's,3.5,83.2,69.6,6,0,85.0
2,3,Japan Women's,4.6,79.4,67.9,5,0,85.0
3,4,Spain Women's,3.8,85.6,68.8,5,0,84.4
4,5,Spain Women's,4.2,84.1,83.8,3,0,80.7
5,6,Germany Women's,2.3,85.1,70.3,6,0,80.0
6,7,Brazil Women's,3.6,84.0,73.6,4,0,79.4
7,8,England Women's,2.3,87.5,71.1,6,1,79.2
8,9,France Women's,3.3,80.3,63.3,6,3,75.1
9,10,Sweden Women's,3.6,74.7,47.5,5,0,73.5


In [45]:
# ==========================================
# SCORE BREAKDOWN
# ==========================================

score_breakdown = group_stage[
    [
        "team",
        "xG_score",
        "pass_accuracy_score",
        "possession_score",
        "goals_scored_score",
        "goals_conceded_score",
        "performance_score"
    ]
].copy()



score_breakdown = score_breakdown.sort_values(
    "performance_score",
    ascending=False
).reset_index(drop=True)



score_breakdown.insert(
    0,
    "Rank",
    range(1, len(score_breakdown) + 1)
)



score_columns = [
    "xG_score",
    "pass_accuracy_score",
    "possession_score",
    "goals_scored_score",
    "goals_conceded_score",
    "performance_score"
]

score_breakdown[score_columns] = (
    score_breakdown[score_columns].round(1)
)



score_breakdown = score_breakdown.rename(columns={
    "team": "Team",
    "xG": "xG (25%)",
    "pass_accuracy_score": "Pass Accuracy Score (25%)",
    "possession_score": "Possession Score (10%)",
    "goals_scored_score": "Goals Scored Score (25%)",
    "goals_conceded_score": "Goals Conceded Score (15%)",
    "performance_score": "Performance Score"
})



score_breakdown

,Rank,Team,xG_score,Pass Accuracy Score (25%),Possession Score (10%),Goals Scored Score (25%),Goals Conceded Score (15%),Performance Score
0,1,Netherlands Women's,91.8,97.2,84.3,100.0,100.0,95.7
1,2,Norway Women's,75.4,87.4,79.0,85.7,100.0,85.0
2,3,Japan Women's,100.0,77.9,76.4,71.4,100.0,85.0
3,4,Spain Women's,81.7,93.3,77.9,71.4,100.0,84.4
4,5,Spain Women's,90.3,89.5,100.0,42.9,100.0,80.7
...,...,...,...,...,...,...,...,...
91,92,Morocco Women's,3.8,36.8,19.9,0.0,14.3,14.3
92,93,Vietnam Women's,0.0,11.7,18.3,0.0,57.1,13.3
93,94,Vietnam Women's,6.0,37.1,15.7,0.0,0.0,12.3
94,95,Philippines Women's,2.4,22.1,21.0,0.0,14.3,10.4
